In [117]:
!pip install duckdb --quiet

In [118]:
import duckdb
import pandas as pd
from IPython.display import display, HTML

df = pd.read_csv('final.csv')

df['discount_flag']   = (df['Discount Applied/Promo Code Used'] == 'Yes').astype(int)
df['full_price_buyer'] = 1 - df['discount_flag']

# Integrity checks
assert (df['discount_flag'] + df['full_price_buyer'] == 1).all(), \
    "Data integrity error: discount_flag + full_price_buyer != 1"

missing_rating_count = int(df['rating_missing'].sum())

con = duckdb.connect()
con.register('df', df)

# **Q1 - Loyal vs. Discount-Only Buyers**
*Identifying genuinely loyal customers vs. those who purchase only when a discount is available.*

### **Q1a - Segment Distribution**

In [119]:
# Brand Purists & Promo Dependent are near-identical on CLV (0.554 vs 0.552), but Brand Purists need zero discount while 
# Promo Dependent score 0.438 dependency / 0.479 loyalty. Same revenue, different risk.
result_q1a = con.sql(""" 
    SELECT
        customer_segment,
        COUNT(*)                                       AS customer_count,
        ROUND(COUNT(*)*100.0/SUM(COUNT(*)) OVER(), 1) AS pct_of_base,
        ROUND(AVG("Purchase Amount (USD)"), 2)         AS avg_spend,
        ROUND(AVG("Previous Purchases"), 2)            AS avg_prev_purchases,
        ROUND(AVG(clv_score), 3)                       AS avg_clv_score,
        ROUND(AVG(loyalty_score), 3)                   AS avg_loyalty_score,
        ROUND(AVG(dependency_score), 3)                AS avg_dependency_score,
        -- Dominant loyalty_tier within each segment (mode approximation)
        MODE("loyalty_tier")                           AS dominant_loyalty_tier,
        -- median used because mean (~258 wks) is severely inflated by outliers
        ROUND(MEDIAN("Tenure in weeks"), 1)            AS median_tenure_weeks
    FROM df
    GROUP BY customer_segment
    ORDER BY avg_clv_score DESC
""").df()
display(result_q1a)

,customer_segment,customer_count,pct_of_base,avg_spend,avg_prev_purchases,avg_clv_score,avg_loyalty_score,avg_dependency_score,dominant_loyalty_tier,median_tenure_weeks
0,Loyal: Brand Purist,1073,28.4,60.73,33.71,0.554,0.767,0.000,High,30.0
1,Loyal: Promo Dependent,831,22.0,59.83,34.53,0.552,0.479,0.438,Mid,35.0
2,Casual: Organic,1079,28.5,59.24,15.58,0.276,0.508,0.000,Mid,52.0
3,Casual: Bargain Hunter,797,21.1,58.67,15.69,0.274,0.210,0.802,Low,52.0


### **Q1b - Validating the Adopted Loyalty Definition (loyalty_score)**

In [120]:
# loyalty_tier (Definition B) shows a clean, monotonic dependency gradient - High=0.049, Mid=0.208, Low=0.533 - confirming it captures
# commitment, not just revenue. CLV/spend stay relatively flat across tiers, showing loyalty_score does not equal to clv_score.
result_q1b = con.sql("""
    SELECT
        loyalty_tier,
        COUNT(*)                                       AS customers,
        ROUND(COUNT(*)*100.0/SUM(COUNT(*)) OVER(), 1)  AS pct_of_base,
        ROUND(AVG(dependency_score), 3)                AS avg_dependency_score,
        ROUND(AVG(clv_score), 3)                       AS avg_clv_score,
        ROUND(AVG("Purchase Amount (USD)"), 2)         AS avg_spend
    FROM df
    GROUP BY loyalty_tier
    ORDER BY loyalty_tier
""").df()
display(result_q1b)

,loyalty_tier,customers,pct_of_base,avg_dependency_score,avg_clv_score,avg_spend
0,High,1233,32.6,0.049,0.530,60.59
1,Low,1272,33.7,0.533,0.306,59.92
2,Mid,1275,33.7,0.208,0.413,58.53


### **Q1c - Why Definition B (Margin Loyalty) Was Adopted**

Deliverable 1 tested two loyalty definitions and adopted Definition B (margin loyalty:
0.7 * engagement + 0.3 * full-price behavior), now stored as `loyalty_score`/`loyalty_tier`.

Definition A (behavioral: engagement + frequency + subscription, equal thirds) was
rejected - its top quartile was 81% discount users and 77% subscribers, meaning it
rewarded discount-access behavior rather than genuine loyalty, and was largely
redundant with `commitment_factor` (r ~ 0.9).

Q1b confirms B behaves as intended: avg_dependency_score falls from low value in the
Low loyalty_tier to high value in the High loyalty_tier - exactly the gradient a
margin-loyalty measure should produce.

# **Q2 - Behavioral Predictors of Customer Value**
*Identifying which behavioral signals - purchase frequency, subscription - predict high CLV.*

### **Q2a - Purchase Frequency vs. Customer Value**

In [121]:
# Bi-Weekly (104/yr) leads on CLV (0.671) and lowest promo dependency (0.103); value declines and dependency rises as frequency drops. 
# Spend/rating stay flat across groups - frequency is the main driver of customer value.
result_q2a = con.sql("""
    SELECT
        "Frequency of Purchases",
        MAX(yearly_purchase_rate)              AS yearly_purchase_rate,
        COUNT(*)                               AS customers,
        ROUND(AVG(clv_score), 3)               AS avg_clv,
        ROUND(AVG(loyalty_score), 3)           AS avg_loyalty_score,
        ROUND(AVG("Purchase Amount (USD)"), 2) AS avg_spend,
        ROUND(AVG(dependency_score), 3)        AS avg_promo_dependency,
        ROUND(AVG("Review Rating"), 2)         AS avg_rating,
        ROUND(AVG("Previous Purchases"), 1)    AS avg_prev_purchases
    FROM df
    GROUP BY "Frequency of Purchases"
    ORDER BY yearly_purchase_rate DESC
""").df()
display(result_q2a)

,Frequency of Purchases,yearly_purchase_rate,customers,avg_clv,avg_loyalty_score,avg_spend,avg_promo_dependency,avg_rating,avg_prev_purchases
0,Bi-Weekly,104,547,0.671,0.516,60.69,0.103,3.71,24.8
1,Weekly,52,539,0.497,0.524,58.97,0.213,3.76,25.8
2,Fortnightly,26,542,0.405,0.512,59.05,0.277,3.76,25.3
3,Monthly,12,553,0.358,0.518,59.33,0.300,3.77,25.3
4,Quarterly,4,1147,0.338,0.525,60.03,0.320,3.75,25.9
5,Annually,1,452,0.284,0.456,59.50,0.331,3.76,20.7


### **Q2b - Subscription Status as Loyalty Signal**

In [122]:
# Insight: clv_score excludes subscription_binary by design.
# The small CLV gap (subscribers vs non) is real but weak. The query below surfaces full_price_rate to show subscription's actual association is with 
# discount usage (r ~ -0.70), not value.
result_q2b = con.sql("""
    SELECT
        "Subscription Status",
        COUNT(*)                               AS customers,
        ROUND(AVG(clv_score), 3)               AS avg_clv,
        ROUND(AVG(loyalty_score), 3)           AS avg_loyalty_score,
        ROUND(AVG(full_price_buyer), 3)        AS full_price_rate,
        ROUND(AVG(dependency_score), 3)        AS avg_dependency_score,
        ROUND(AVG("Purchase Amount (USD)"), 2) AS avg_spend,
        ROUND(AVG("Review Rating"), 2)         AS avg_rating,
        ROUND(AVG("Previous Purchases"), 1)    AS avg_prev_purchases,
        ROUND(MEDIAN("Tenure in weeks"), 1)    AS median_tenure_weeks
    FROM df
    GROUP BY "Subscription Status"
    ORDER BY avg_clv DESC
""").df()
display(result_q2b)

,Subscription Status,customers,avg_clv,avg_loyalty_score,full_price_rate,avg_dependency_score,avg_spend,avg_rating,avg_prev_purchases,median_tenure_weeks
0,Yes,1019,0.421,0.352,0.000,0.611,59.49,3.75,25.6,42.0
1,No,2761,0.413,0.572,0.779,0.138,59.74,3.75,24.6,39.0


### **Q2c - Frequency x Subscription: Combined Value Profile**

In [123]:
# Highest CLV cell is Bi-Weekly+Subscribed (0.688), but reflects  frequency/engagement (clv_score components) not subscription itself (excluded from the formula) 
# Bi-Weekly is small; Weekly+Subscribed and Fortnightly+Subscribed may balance value and audience size better.
result_q2c = con.sql("""
    SELECT
        "Frequency of Purchases",
        "Subscription Status",
        COUNT(*)                               AS customers,
        ROUND(AVG(clv_score), 3)               AS avg_clv,
        ROUND(AVG(loyalty_score), 3)           AS avg_loyalty_score,
        ROUND(AVG(dependency_score), 3)        AS avg_promo_dependency,
        ROUND(AVG("Review Rating"), 2)         AS avg_rating,
        ROUND(AVG(Age), 1)                     AS avg_age,
        ROUND(AVG("Purchase Amount (USD)"), 2) AS avg_spend,
        ROUND(AVG("Previous Purchases"), 2)    AS avg_prev_purchases
    FROM df
    GROUP BY "Frequency of Purchases", "Subscription Status"
    ORDER BY avg_clv DESC
""").df()
display(result_q2c)

,Frequency of Purchases,Subscription Status,customers,avg_clv,avg_loyalty_score,avg_promo_dependency,avg_rating,avg_age,avg_spend,avg_prev_purchases
0,Bi-Weekly,Yes,140,0.688,0.372,0.234,3.70,44.5,60.14,27.07
1,Bi-Weekly,No,407,0.666,0.565,0.058,3.71,42.8,60.88,24.00
2,Weekly,Yes,157,0.505,0.367,0.490,3.78,45.9,59.10,26.69
3,Weekly,No,382,0.494,0.588,0.099,3.75,44.1,58.92,25.39
4,Fortnightly,Yes,153,0.407,0.358,0.623,3.75,44.9,57.76,26.04
5,Fortnightly,No,389,0.404,0.573,0.141,3.76,43.1,59.56,24.97
6,Monthly,Yes,149,0.363,0.355,0.693,3.76,43.7,59.11,25.87
7,Monthly,No,404,0.357,0.578,0.155,3.78,44.5,59.41,25.06
8,Quarterly,Yes,294,0.344,0.360,0.728,3.79,44.1,60.85,26.22
9,Quarterly,No,853,0.337,0.582,0.179,3.73,44.0,59.75,25.77


# **Q3 - Geographic & Product Opportunity**
*Identifying which states, product categories, and customer demographics represent genuine brand pull versus discount-driven demand. Geographic opportunity is evaluated using an Organic Demand Proxy, while product and demographic-level opportunities are explored later.*

### **Q3a - State-Level Performance (Organic Demand Proxy)**

In [124]:
# organic_demand_proxy = avg_spend x (1 - dependency_score), from final.csv 
# Top states: Arizona (54.17), Alaska (51.74), Tennessee (49.43) - high spend + low promo reliance
# All 50 states clear the >=50 customer threshold (min: Kansas, 58), so rankings are fully comparable
result_q3a = con.sql("""
SELECT
    Location                                          AS state,
    COUNT(*)                                          AS customers,
    ROUND(SUM("Purchase Amount (USD)"), 0)            AS total_revenue,
    ROUND(AVG("Purchase Amount (USD)"), 2)            AS avg_spend,
    ROUND(AVG(clv_score), 3)                          AS avg_clv,
    ROUND(AVG(loyalty_score), 3)                      AS avg_loyalty_score,
    ROUND(AVG(dependency_score), 3)                   AS avg_promo_dependency,
    ROUND(AVG(full_price_buyer), 3)                   AS full_price_rate,
    ROUND(AVG("Purchase Amount (USD)" * (1 - dependency_score)), 2) AS organic_demand_proxy
FROM df
GROUP BY Location
HAVING COUNT(*) >= 50
ORDER BY organic_demand_proxy DESC
""").df()
display(result_q3a)

,state,customers,total_revenue,avg_spend,avg_clv,avg_loyalty_score,avg_promo_dependency,full_price_rate,organic_demand_proxy
0,Arizona,62,4116.0,66.39,0.476,0.585,0.184,0.661,54.17
1,Alaska,71,4795.0,67.54,0.455,0.564,0.251,0.592,51.74
2,Tennessee,76,4693.0,61.75,0.439,0.547,0.226,0.645,49.43
3,Illinois,88,5339.0,60.67,0.459,0.532,0.217,0.580,48.91
4,New Mexico,78,4815.0,61.73,0.425,0.516,0.244,0.564,48.91
5,Pennsylvania,74,4926.0,66.57,0.463,0.544,0.277,0.554,48.20
6,Michigan,67,4171.0,62.25,0.413,0.528,0.253,0.597,48.15
7,Kansas,58,3160.0,54.48,0.403,0.537,0.125,0.793,48.00
8,North Dakota,82,5153.0,62.84,0.405,0.484,0.265,0.537,47.30
9,Virginia,74,4567.0,61.72,0.407,0.500,0.251,0.622,46.96


### **Q3b - Demographic Opportunity Analysis**

In [125]:
# Male 46-55 has highest avg_clv (0.433). But Gender is confounded with discount usage - all females show full_price_rate=1.0 / dependency=0,
# all discount/subscription activity is male-only. 
# So gender comparisons mostly reflect this confound, not real demographic signal. Age adds little vs. behavioral variables.
result_q3b = con.sql("""
SELECT
    Gender,
    CASE
        WHEN Age BETWEEN 18 AND 25 THEN '18-25'
        WHEN Age BETWEEN 26 AND 35 THEN '26-35'
        WHEN Age BETWEEN 36 AND 45 THEN '36-45'
        WHEN Age BETWEEN 46 AND 55 THEN '46-55'
        ELSE '56+'
    END                                        AS age_group,
    COUNT(*)                                   AS customers,
    ROUND(AVG("Purchase Amount (USD)"), 2)     AS avg_spend,
    ROUND(AVG("Previous Purchases"), 2)        AS avg_previous_purchases,
    ROUND(AVG(clv_score), 3)                   AS avg_clv,
    ROUND(AVG(loyalty_score), 3)               AS avg_loyalty_score,
    ROUND(AVG(dependency_score), 3)            AS avg_promo_dependency,
    ROUND(AVG(full_price_buyer), 3)            AS full_price_rate
FROM df
GROUP BY Gender, age_group
ORDER BY avg_clv DESC, avg_previous_purchases DESC
""").df()
display(result_q3b)

,Gender,age_group,customers,avg_spend,avg_previous_purchases,avg_clv,avg_loyalty_score,avg_promo_dependency,full_price_rate
0,Male,46-55,506,61.45,25.91,0.433,0.463,0.388,0.358
1,Female,26-35,229,62.11,24.10,0.430,0.630,0.000,1.000
2,Male,56+,761,58.47,26.64,0.417,0.478,0.385,0.371
3,Female,56+,344,60.35,25.19,0.416,0.646,0.000,1.000
4,Male,18-25,358,60.15,24.15,0.413,0.443,0.387,0.374
5,Male,26-35,476,58.48,23.84,0.410,0.430,0.404,0.347
6,Female,36-45,233,59.33,23.58,0.409,0.623,0.000,1.000
7,Female,18-25,158,60.69,22.28,0.408,0.604,0.000,1.000
8,Male,36-45,470,59.11,24.81,0.406,0.456,0.392,0.385
9,Female,46-55,245,58.84,24.23,0.402,0.632,0.000,1.000


### **Q3c - Item Purchased: Value & Promo Dependency by Product**

In [126]:
# Q3c-i: All items ranked by organic purchase strength

# organic_purchase_strength = clv_score x (1-dependency_score)
# Top items combine high CLV with low dependency; bottom items lean toward promo-driven purchases. 
# Product choice is a stronger value indicator than seasonality.
result_q3c_i = con.sql("""
SELECT
    "Item Purchased",
    Category,
    COUNT(*)                                     AS customers,
    ROUND(AVG("Purchase Amount (USD)"), 2)        AS avg_spend,
    ROUND(AVG(clv_score), 3)                     AS avg_clv,
    ROUND(AVG(loyalty_score), 3)                 AS avg_loyalty_score,
    ROUND(AVG(dependency_score), 3)              AS avg_promo_dependency,
    ROUND(AVG(full_price_buyer), 3)              AS full_price_rate,
    ROUND(AVG(clv_score * (1 - dependency_score)), 3) AS organic_purchase_strength
FROM df
GROUP BY "Item Purchased", Category
ORDER BY organic_purchase_strength DESC
""").df()
display(result_q3c_i)

,Item Purchased,Category,customers,avg_spend,avg_clv,avg_loyalty_score,avg_promo_dependency,full_price_rate,organic_purchase_strength
0,Jewelry,Accessories,165,58.48,0.456,0.560,0.250,0.558,0.364
1,Blouse,Clothing,167,61.10,0.436,0.559,0.221,0.653,0.359
2,Gloves,Accessories,134,60.28,0.435,0.525,0.266,0.567,0.344
3,Shorts,Clothing,151,59.75,0.437,0.521,0.259,0.570,0.339
4,Sunglasses,Accessories,158,60.06,0.422,0.525,0.248,0.589,0.333
5,Boots,Footwear,140,62.41,0.442,0.514,0.286,0.529,0.333
6,Shirt,Clothing,159,61.09,0.424,0.521,0.248,0.585,0.332
7,Sweater,Clothing,159,57.13,0.431,0.500,0.283,0.516,0.327
8,Scarf,Accessories,152,60.84,0.423,0.535,0.258,0.586,0.327
9,Skirt,Clothing,154,58.84,0.410,0.522,0.242,0.617,0.326


In [127]:
# Q3c-ii: Item preference by customer segment (top 5 items per segment)

# Top items differ meaningfully by segment - Brand Purists' top items show low avg_promo_dep, while Bargain Hunters' top items show high avg_promo_dep.
# Product preferences track segment behavior, not just overall popularity.
result_q3c_ii = con.sql("""
WITH ranked AS (
    SELECT
        customer_segment,
        "Item Purchased",
        COUNT(*)                               AS purchase_count,
        ROUND(AVG(clv_score), 3)               AS avg_clv,
        ROUND(AVG(dependency_score), 3)        AS avg_promo_dep,
        ROW_NUMBER() OVER (
            PARTITION BY customer_segment
            ORDER BY COUNT(*) DESC
        ) AS item_rank
    FROM df
    GROUP BY customer_segment, "Item Purchased"
)
SELECT *
FROM ranked
WHERE item_rank <= 5
ORDER BY customer_segment, item_rank, "Item Purchased" ASC
""").df()
display(result_q3c_ii)

,customer_segment,Item Purchased,purchase_count,avg_clv,avg_promo_dep,item_rank
0,Casual: Bargain Hunter,Pants,49,0.263,0.810,1
1,Casual: Bargain Hunter,Belt,39,0.288,0.785,2
2,Casual: Bargain Hunter,Hat,38,0.288,0.787,3
3,Casual: Bargain Hunter,Jacket,37,0.235,0.826,4
4,Casual: Bargain Hunter,Handbag,37,0.240,0.833,5
5,Casual: Organic,Sandals,53,0.275,0.000,1
6,Casual: Organic,Jacket,53,0.249,0.000,2
7,Casual: Organic,Socks,53,0.259,0.000,3
8,Casual: Organic,T-shirt,51,0.280,0.000,4
9,Casual: Organic,Skirt,49,0.282,0.000,5


In [128]:
# Q3c-iii: Item x Season - which items are seasonal acquisition vs. year-round retention drivers?

# Item x Season counts are fairly even (no missing combos), but off-season purchases (e.g. winter items in summer) show higher avg_promo_dep
# than in-season ones - likely clearance/discount-driven rather than retention behavior.
result_q3c_iii = con.sql("""
SELECT
    "Item Purchased",
    Season,
    COUNT(*)                               AS purchase_count,
    ROUND(AVG(clv_score), 3)               AS avg_clv,
    ROUND(AVG(dependency_score), 3)        AS avg_promo_dep,
    ROUND(AVG("Previous Purchases"), 1)    AS avg_prev_purchases
FROM df
GROUP BY "Item Purchased", Season
ORDER BY "Item Purchased", Season
""").df()
display(result_q3c_iii)

,Item Purchased,Season,purchase_count,avg_clv,avg_promo_dep,avg_prev_purchases
0,Backpack,Fall,33,0.384,0.328,20.8
1,Backpack,Spring,39,0.404,0.260,24.0
2,Backpack,Summer,42,0.409,0.237,23.5
3,Backpack,Winter,25,0.438,0.279,23.9
4,Belt,Fall,39,0.393,0.287,24.2
...,...,...,...,...,...,...
95,Sweater,Winter,42,0.469,0.240,27.5
96,T-shirt,Fall,39,0.397,0.259,24.4
97,T-shirt,Spring,36,0.462,0.269,26.1
98,T-shirt,Summer,28,0.339,0.242,18.4


# **Q4 - Promotional Dependency & Sunset Strategy**
*Measuring true promo dependency per segment, identifying sunset candidates, and determining when each segment needs promotions most.*

### **Q4a - Promo Dependent vs Brand Purist: Behavioural Equivalence Test**

In [129]:
# Q4a-i: Head-to-head comparison of Loyal segments across every behavioural dimension
# Goal: establish that Promo Dependent behaviour mirrors Brand Purist behaviour, meaning the discount is a habit not a necessity - justifying the Premium merge
result_q4a = con.sql("""
SELECT
    customer_segment,
    COUNT(*)                                        AS customers,
    -- Purchase depth
    ROUND(AVG("Previous Purchases"), 2)             AS avg_prev_purchases,
    ROUND(MEDIAN("Previous Purchases"), 1)          AS median_prev_purchases,
    -- Spend
    ROUND(AVG("Purchase Amount (USD)"), 2)          AS avg_spend,
    ROUND(MEDIAN("Purchase Amount (USD)"), 2)       AS median_spend,
    ROUND(STDDEV("Purchase Amount (USD)"), 2)       AS std_spend,
    -- Purchase cadence
    ROUND(AVG(yearly_purchase_rate), 1)             AS avg_yearly_txns,
    MODE("Frequency of Purchases")                  AS dominant_frequency,
    -- Loyalty & consistency
    ROUND(AVG(consistency_score), 3)                AS avg_consistency,
    ROUND(AVG(loyalty_score), 3)                    AS avg_loyalty,
    ROUND(AVG(clv_score), 3)                        AS avg_clv,
    -- Promo behaviour
    ROUND(AVG(dependency_score), 3)                 AS avg_dependency,
    ROUND(AVG(full_price_buyer), 3)                 AS full_price_rate,
    -- Satisfaction
    ROUND(AVG("Review Rating"), 3)                  AS avg_rating,
    MODE(satisfaction_tier)                         AS dominant_satisfaction,
    -- Tenure
    ROUND(MEDIAN("Tenure in weeks"), 1)             AS median_tenure_weeks
FROM df
WHERE customer_segment IN ('Loyal: Brand Purist', 'Loyal: Promo Dependent')
GROUP BY customer_segment
ORDER BY customer_segment
""").df()
display(result_q4a)

,customer_segment,customers,avg_prev_purchases,median_prev_purchases,avg_spend,median_spend,std_spend,avg_yearly_txns,dominant_frequency,avg_consistency,avg_loyalty,avg_clv,avg_dependency,full_price_rate,avg_rating,dominant_satisfaction,median_tenure_weeks
0,Loyal: Brand Purist,1073,33.71,37.0,60.73,61.0,23.62,48.0,Bi-Weekly,0.562,0.767,0.554,0.000,1.0,3.756,Neutral,30.0
1,Loyal: Promo Dependent,831,34.53,37.0,59.83,61.0,23.91,46.2,Bi-Weekly,0.562,0.479,0.552,0.438,0.0,3.729,Neutral,35.0


In [130]:
# Q4a-ii: Frequency distribution breakdown side by side
# Shows whether both segments buy at the same cadence
result_q4a_freq = con.sql("""
SELECT
    customer_segment,
    "Frequency of Purchases",
    COUNT(*)                                            AS customers,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (
        PARTITION BY customer_segment
    ), 1)                                               AS pct_within_segment
FROM df
WHERE customer_segment IN ('Loyal: Brand Purist', 'Loyal: Promo Dependent')
GROUP BY customer_segment, "Frequency of Purchases"
ORDER BY customer_segment,
         MAX(yearly_purchase_rate) DESC
""").df()
display(result_q4a_freq)

,customer_segment,Frequency of Purchases,customers,pct_within_segment
0,Loyal: Brand Purist,Bi-Weekly,321,29.9
1,Loyal: Brand Purist,Weekly,230,21.4
2,Loyal: Brand Purist,Fortnightly,149,13.9
3,Loyal: Brand Purist,Monthly,114,10.6
4,Loyal: Brand Purist,Quarterly,228,21.2
5,Loyal: Brand Purist,Annually,31,2.9
6,Loyal: Promo Dependent,Bi-Weekly,226,27.2
7,Loyal: Promo Dependent,Weekly,185,22.3
8,Loyal: Promo Dependent,Fortnightly,135,16.2
9,Loyal: Promo Dependent,Monthly,88,10.6


In [131]:
# Q4a-iii: CLV tier distribution - are both segments equally distributed across tiers?
result_q4a_clv = con.sql("""
SELECT
    customer_segment,
    clv_tier,
    COUNT(*)                                            AS customers,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (
        PARTITION BY customer_segment
    ), 1)                                               AS pct_within_segment,
    ROUND(AVG(loyalty_score), 3)                        AS avg_loyalty,
    ROUND(AVG(consistency_score), 3)                    AS avg_consistency
FROM df
WHERE customer_segment IN ('Loyal: Brand Purist', 'Loyal: Promo Dependent')
GROUP BY customer_segment, clv_tier
ORDER BY customer_segment, clv_tier DESC
""").df()
display(result_q4a_clv)

,customer_segment,clv_tier,customers,pct_within_segment,avg_loyalty,avg_consistency
0,Loyal: Brand Purist,Mid,363,33.8,0.760,0.454
1,Loyal: Brand Purist,Low,9,0.8,0.637,0.387
2,Loyal: Brand Purist,High,701,65.3,0.773,0.620
3,Loyal: Promo Dependent,Mid,287,34.5,0.455,0.449
4,Loyal: Promo Dependent,Low,8,1.0,0.362,0.384
5,Loyal: Promo Dependent,High,536,64.5,0.493,0.625


### **Q4b - Effective Discount Value: Credit Ceiling Calibration**

In [132]:
# Q4b-i: Cross-segment spend comparison
# Brand Purist = full-price baseline
# Promo Dependent = discounted baseline
# The gap between them is the monetary value of the discount being given away
# This gap becomes the ceiling for store credit value
result_q4b_i = con.sql("""
SELECT
    customer_segment,
    COUNT(*)                                        AS customers,
    ROUND(AVG("Purchase Amount (USD)"), 2)          AS avg_spend,
    ROUND(MEDIAN("Purchase Amount (USD)"), 2)       AS median_spend,
    ROUND(STDDEV("Purchase Amount (USD)"), 2)       AS std_spend,
    ROUND(MIN("Purchase Amount (USD)"), 2)          AS min_spend,
    ROUND(MAX("Purchase Amount (USD)"), 2)          AS max_spend
FROM df
WHERE customer_segment IN (
    'Loyal: Brand Purist',
    'Loyal: Promo Dependent',
    'Casual: Organic',
    'Casual: Bargain Hunter'
)
GROUP BY customer_segment
ORDER BY avg_spend DESC
""").df()
display(result_q4b_i)

,customer_segment,customers,avg_spend,median_spend,std_spend,min_spend,max_spend
0,Loyal: Brand Purist,1073,60.73,61.0,23.62,20,100
1,Loyal: Promo Dependent,831,59.83,61.0,23.91,20,100
2,Casual: Organic,1079,59.24,58.0,23.90,20,100
3,Casual: Bargain Hunter,797,58.67,59.0,23.38,20,100


In [133]:
# Q4b-ii: Compute credit ceiling values directly
# Loyal gap  -> Premium membership credit ceiling
# Casual gap -> General base credit ceiling
result_q4b_ii = con.sql("""
WITH seg_spend AS (
    SELECT
        customer_segment,
        AVG("Purchase Amount (USD)") AS avg_spend
    FROM df
    GROUP BY customer_segment
),
pivoted AS (
    SELECT
        MAX(CASE WHEN customer_segment = 'Loyal: Brand Purist'    THEN avg_spend END) AS purist_avg,
        MAX(CASE WHEN customer_segment = 'Loyal: Promo Dependent' THEN avg_spend END) AS promodep_avg,
        MAX(CASE WHEN customer_segment = 'Casual: Organic'        THEN avg_spend END) AS organic_avg,
        MAX(CASE WHEN customer_segment = 'Casual: Bargain Hunter' THEN avg_spend END) AS bargain_avg
    FROM seg_spend
)
SELECT
    -- Loyal tier
    ROUND(purist_avg, 2)                                            AS loyal_fullprice_baseline,
    ROUND(promodep_avg, 2)                                          AS loyal_discounted_avg,
    ROUND(purist_avg - promodep_avg, 2)                             AS loyal_discount_value_per_txn,
    ROUND((purist_avg - promodep_avg) / purist_avg * 100, 2)       AS loyal_effective_discount_rate_pct,
    -- Casual tier
    ROUND(organic_avg, 2)                                           AS casual_fullprice_baseline,
    ROUND(bargain_avg, 2)                                           AS casual_discounted_avg,
    ROUND(organic_avg - bargain_avg, 2)                             AS casual_discount_value_per_txn,
    ROUND((organic_avg - bargain_avg) / organic_avg * 100, 2)      AS casual_effective_discount_rate_pct,
    -- Credit value recommendations
    -- General base: match casual ceiling (feel equivalent to discount)
    -- Premium: set at 50% of loyal ceiling (non-monetary perks compensate the rest)
    ROUND((organic_avg - bargain_avg) / organic_avg * 100, 2)      AS general_credit_ceiling_pct,
    ROUND((purist_avg - promodep_avg) / purist_avg * 100 * 0.5, 2) AS premium_credit_ceiling_pct
FROM pivoted
""").df()
display(result_q4b_ii)

,loyal_fullprice_baseline,loyal_discounted_avg,loyal_discount_value_per_txn,loyal_effective_discount_rate_pct,casual_fullprice_baseline,casual_discounted_avg,casual_discount_value_per_txn,casual_effective_discount_rate_pct,general_credit_ceiling_pct,premium_credit_ceiling_pct
0,60.73,59.83,0.89,1.47,59.24,58.67,0.57,0.96,0.96,0.73


In [153]:
# Sanity check - flag if gaps are near zero or negative
print("\n=== Sanity Check ===")
print("If loyal_discount_value_per_txn or casual_discount_value_per_txn is near 0")
print("or negative, the spend gap proxy is not valid for that tier.")
print("Loyal gap drives the Premium credit ceiling - casual gap is supplementary.")


=== Sanity Check ===
If loyal_discount_value_per_txn or casual_discount_value_per_txn is near 0
or negative, the spend gap proxy is not valid for that tier.
Loyal gap drives the Premium credit ceiling - casual gap is supplementary.


### **Q4c - Purchase Rhythm: Credit Expiry Window Calibration**

In [135]:
# Q4c-i: Purchase interval in days by segment
# Expiry window = 2x median interval (two full natural purchase cycles)
# Enough time to redeem naturally; not so long that urgency disappears
result_q4c_i = con.sql("""
SELECT
    customer_segment,
    "Frequency of Purchases",
    COUNT(*)                                            AS customers,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (
        PARTITION BY customer_segment
    ), 1)                                               AS pct_within_segment,
    -- Map frequency to interval in days
    CASE "Frequency of Purchases"
        WHEN 'Bi-Weekly'    THEN 3.5
        WHEN 'Weekly'       THEN 7
        WHEN 'Fortnightly'  THEN 14
        WHEN 'Monthly'      THEN 30
        WHEN 'Quarterly'    THEN 91
        WHEN 'Annually'     THEN 365
    END                                                 AS interval_days,
    -- Recommended expiry = 2x interval
    CASE "Frequency of Purchases"
        WHEN 'Bi-Weekly'    THEN 7
        WHEN 'Weekly'       THEN 14
        WHEN 'Fortnightly'  THEN 28
        WHEN 'Monthly'      THEN 60
        WHEN 'Quarterly'    THEN 182
        WHEN 'Annually'     THEN 365
    END                                                 AS recommended_expiry_days
FROM df
WHERE customer_segment IN ('Loyal: Brand Purist', 'Loyal: Promo Dependent',
                           'Casual: Organic',      'Casual: Bargain Hunter')
GROUP BY customer_segment, "Frequency of Purchases"
ORDER BY customer_segment,
         MAX(yearly_purchase_rate) DESC
""").df()
display(result_q4c_i)

,customer_segment,Frequency of Purchases,customers,pct_within_segment,interval_days,recommended_expiry_days
0,Casual: Bargain Hunter,Weekly,49,6.1,7.0,14
1,Casual: Bargain Hunter,Fortnightly,108,13.6,14.0,28
2,Casual: Bargain Hunter,Monthly,149,18.7,30.0,60
3,Casual: Bargain Hunter,Quarterly,333,41.8,91.0,182
4,Casual: Bargain Hunter,Annually,158,19.8,365.0,365
5,Casual: Organic,Weekly,75,7.0,7.0,14
6,Casual: Organic,Fortnightly,150,13.9,14.0,28
7,Casual: Organic,Monthly,202,18.7,30.0,60
8,Casual: Organic,Quarterly,420,38.9,91.0,182
9,Casual: Organic,Annually,232,21.5,365.0,365


In [136]:
# Q4c-ii: Segment-level weighted median expiry window
# Weights by customer count so the dominant frequency drives the recommendation
result_q4c_ii = con.sql("""
WITH intervals AS (
    SELECT
        customer_segment,
        CASE "Frequency of Purchases"
            WHEN 'Bi-Weekly'   THEN 3.5
            WHEN 'Weekly'      THEN 7
            WHEN 'Fortnightly' THEN 14
            WHEN 'Monthly'     THEN 30
            WHEN 'Quarterly'   THEN 91
            WHEN 'Annually'    THEN 365
        END AS interval_days
    FROM df
    WHERE customer_segment IN (
        'Loyal: Brand Purist', 'Loyal: Promo Dependent',
        'Casual: Organic',     'Casual: Bargain Hunter'
    )
)
SELECT
    customer_segment,
    ROUND(MEDIAN(interval_days), 1)         AS median_interval_days,
    ROUND(AVG(interval_days), 1)            AS avg_interval_days,
    ROUND(MEDIAN(interval_days) * 2, 0)     AS recommended_expiry_days
FROM intervals
GROUP BY customer_segment
ORDER BY median_interval_days ASC
""").df()
display(result_q4c_ii)

,customer_segment,median_interval_days,avg_interval_days,recommended_expiry_days
0,Loyal: Brand Purist,7.0,37.6,14.0
1,Loyal: Promo Dependent,14.0,39.8,28.0
2,Casual: Bargain Hunter,91.0,118.3,182.0
3,Casual: Organic,91.0,122.0,182.0


### **Q4d - Earn Rate Calibration: Credits Per Purchase Cycle**

In [137]:
# Q4d: How many credits does a customer earn per cycle at different earn rates?
# Logic: earn rate (credits per $ spent) * avg spend = credits earned per purchase
# Target: credits earned in ONE cycle ≈ minimum redemption threshold
# So customer can redeem within their natural purchase rhythm
result_q4d = con.sql("""
WITH base AS (
    SELECT
        customer_segment,
        ROUND(AVG("Purchase Amount (USD)"), 2)  AS avg_spend_per_txn,
        ROUND(AVG(yearly_purchase_rate), 1)     AS avg_yearly_txns,
        ROUND(MEDIAN("Tenure in weeks"), 1)     AS median_tenure_weeks
    FROM df
    WHERE customer_segment IN (
        'Loyal: Brand Purist', 'Loyal: Promo Dependent',
        'Casual: Organic',     'Casual: Bargain Hunter'
    )
    GROUP BY customer_segment
)
SELECT
    customer_segment,
    avg_spend_per_txn,
    avg_yearly_txns,
    -- Credits earned per purchase at 1 credit per $1 spent
    ROUND(avg_spend_per_txn * 1.0, 2)          AS credits_per_txn_rate_1,
    -- Credits earned per purchase at 0.5 credits per $1 (lower earn, premium feel)
    ROUND(avg_spend_per_txn * 0.5, 2)          AS credits_per_txn_rate_0_5,
    -- Annual credit accumulation at each rate
    ROUND(avg_spend_per_txn * 1.0 * avg_yearly_txns, 0)    AS annual_credits_rate_1,
    ROUND(avg_spend_per_txn * 0.5 * avg_yearly_txns, 0)    AS annual_credits_rate_0_5,
    -- $ value of annual credits at each rate (1 credit = $0.01)
    ROUND(avg_spend_per_txn * 1.0 * avg_yearly_txns * 0.01, 2)  AS annual_credit_value_rate_1,
    ROUND(avg_spend_per_txn * 0.5 * avg_yearly_txns * 0.01, 2)  AS annual_credit_value_rate_0_5
FROM base
ORDER BY avg_yearly_txns DESC
""").df()
display(result_q4d)

,customer_segment,avg_spend_per_txn,avg_yearly_txns,credits_per_txn_rate_1,credits_per_txn_rate_0_5,annual_credits_rate_1,annual_credits_rate_0_5,annual_credit_value_rate_1,annual_credit_value_rate_0_5
0,Loyal: Brand Purist,60.73,48.0,60.73,30.37,2915.0,1458.0,29.15,14.58
1,Loyal: Promo Dependent,59.83,46.2,59.83,29.92,2764.0,1382.0,27.64,13.82
2,Casual: Organic,59.24,11.2,59.24,29.62,663.0,332.0,6.63,3.32
3,Casual: Bargain Hunter,58.67,10.8,58.67,29.34,634.0,317.0,6.34,3.17


### **Q4e - Margin Recovery: Financial Case for the Credit System**

In [138]:
# Q4e: Projected margin recovery under the store credit system
# Sunset candidates = Loyal: Promo Dependent, High + Mid CLV, >= 25 prev purchases (median) 
# Recovery = (full-price baseline - discounted avg) x customer count
# Three scenarios: optimistic (0% churn), base case (15% churn), conservative (25% churn)
result_q4e = con.sql("""
WITH baselines AS (
    SELECT
        AVG(CASE WHEN customer_segment = 'Loyal: Brand Purist'
            THEN "Purchase Amount (USD)" END)           AS purist_spend,
        AVG(CASE WHEN customer_segment = 'Loyal: Promo Dependent'
            THEN "Purchase Amount (USD)" END)           AS promodep_spend
    FROM df
),
discount_value AS (
    SELECT
        ROUND(AVG(purist_spend) - AVG(promodep_spend), 2)  AS discount_per_txn
    FROM baselines
),
candidates AS (
    SELECT
        clv_tier,
        COUNT(*)                                            AS sunset_candidates,
        ROUND(AVG("Purchase Amount (USD)"), 2)              AS avg_spend,
        ROUND(AVG(yearly_purchase_rate), 1)                 AS avg_yearly_txns,
        ROUND(AVG(loyalty_score), 3)                        AS avg_loyalty
    FROM df
    WHERE customer_segment = 'Loyal: Promo Dependent'
      AND clv_tier IN ('High', 'Mid')
      AND "Previous Purchases" >= 25
    GROUP BY clv_tier
)
SELECT
    c.clv_tier,
    c.sunset_candidates,
    c.avg_spend,
    c.avg_yearly_txns,
    c.avg_loyalty,
    ROUND(d.discount_per_txn, 2)                                        AS discount_value_per_txn,
    -- Annual margin recovered per customer = discount_per_txn x yearly txns
    ROUND(d.discount_per_txn * c.avg_yearly_txns, 2)                   AS margin_per_customer_annual,
    -- Total pool
    ROUND(d.discount_per_txn * c.avg_yearly_txns * c.sunset_candidates, 0)
                                                                        AS total_recovery_optimistic,
    -- Base case: 15% of candidates churn after sunset
    ROUND(d.discount_per_txn * c.avg_yearly_txns * c.sunset_candidates * 0.85, 0)
                                                                        AS total_recovery_base_case,
    -- Conservative: 25% churn
    ROUND(d.discount_per_txn * c.avg_yearly_txns * c.sunset_candidates * 0.75, 0)
                                                                        AS total_recovery_conservative
FROM candidates c
CROSS JOIN discount_value d
ORDER BY c.clv_tier DESC
""").df()
display(result_q4e)

,clv_tier,sunset_candidates,avg_spend,avg_yearly_txns,avg_loyalty,discount_value_per_txn,margin_per_customer_annual,total_recovery_optimistic,total_recovery_base_case,total_recovery_conservative
0,Mid,230,42.08,14.9,0.525,0.89,13.26,3050.0,2593.0,2288.0
1,High,432,69.28,47.5,0.563,0.89,42.28,18263.0,15523.0,13697.0


In [139]:
# Q4e-ii: Grand total across both CLV tiers
result_q4e_total = con.sql("""
WITH baselines AS (
    SELECT
        AVG(CASE WHEN customer_segment = 'Loyal: Brand Purist'
            THEN "Purchase Amount (USD)" END)           AS purist_spend,
        AVG(CASE WHEN customer_segment = 'Loyal: Promo Dependent'
            THEN "Purchase Amount (USD)" END)           AS promodep_spend
    FROM df
),
discount_value AS (
    SELECT purist_spend - promodep_spend AS discount_per_txn
    FROM baselines
),
candidates AS (
    SELECT
        COUNT(*)                            AS total_candidates,
        AVG("Purchase Amount (USD)")        AS avg_spend,
        AVG(yearly_purchase_rate)           AS avg_yearly_txns
    FROM df
    WHERE customer_segment = 'Loyal: Promo Dependent'
      AND clv_tier IN ('High', 'Mid')
      AND "Previous Purchases" >= 25
)
SELECT
    c.total_candidates,
    ROUND(d.discount_per_txn, 2)                                            AS discount_per_txn,
    ROUND(d.discount_per_txn * c.avg_yearly_txns * c.total_candidates, 0)  AS optimistic,
    ROUND(d.discount_per_txn * c.avg_yearly_txns * c.total_candidates * 0.85, 0) AS base_case,
    ROUND(d.discount_per_txn * c.avg_yearly_txns * c.total_candidates * 0.75, 0) AS conservative
FROM candidates c
CROSS JOIN discount_value d
""").df()
display(result_q4e_total)

,total_candidates,discount_per_txn,optimistic,base_case,conservative
0,662,0.89,21394.0,18185.0,16045.0


# **Q5 - Ideal Customer Profile**
*A data-backed description of the brand's most valuable customer type - incorporating age, purchase habits, payment preferences, and satisfaction - specific enough for a marketing team to use for targeting decisions today.*

### **Q5a - Ideal Customer Profile**

In [140]:
# Insight: Ideal customer = full-price, repeat buyer, longer tenure, above-average spend.
# Skews older, mid-to-large sizes, prefers accessories/staples over seasonal items.
# Mostly Neutral/Highly Satisfied -> stable, self-sustaining revenue base, best acquisition template.

# Caveat: All subscribers excluded (every subscriber has discount_flag=1) and all females
# appear in ICP (every female has full_price_buyer=1) - both are dataset structure effects,
# not evidence of real non-subscriber/female loyalty advantage.
result_q5a = con.sql("""
    SELECT
        Gender,
        "Frequency of Purchases",
        "Subscription Status",
        "Payment Method",
        MODE(satisfaction_tier)                AS dominant_satisfaction_tier,
        ROUND(AVG(Age), 0)                     AS avg_age,
        ROUND(AVG("Review Rating"), 2)         AS avg_rating,
        ROUND(AVG("Purchase Amount (USD)"), 0) AS avg_spend,
        ROUND(AVG("Previous Purchases"), 0)    AS avg_prev_purchases,
        ROUND(AVG(clv_score), 3)               AS avg_clv,
        ROUND(AVG(loyalty_score), 3)           AS avg_loyalty_score,
        ROUND(AVG(dependency_score), 3)        AS avg_promo_dep,
        COUNT(*)                               AS customer_count
    FROM df
    WHERE
        full_price_buyer = 1
        AND clv_tier = 'High'
        AND consistent = 1
    GROUP BY
        Gender,
        "Frequency of Purchases",
        "Subscription Status",
        "Payment Method"
    ORDER BY customer_count DESC
""").df()
display(result_q5a)

,Gender,Frequency of Purchases,Subscription Status,Payment Method,dominant_satisfaction_tier,avg_age,avg_rating,avg_spend,avg_prev_purchases,avg_clv,avg_loyalty_score,avg_promo_dep,customer_count
0,Female,Bi-Weekly,No,Cash,Neutral,47.0,3.82,60.0,22.0,0.645,0.596,0.0,31
1,Female,Bi-Weekly,No,Bank Transfer,Highly Satisfied,39.0,3.60,65.0,26.0,0.696,0.661,0.0,30
2,Female,Bi-Weekly,No,Debit Card,Neutral,47.0,3.65,72.0,27.0,0.726,0.675,0.0,26
3,Female,Bi-Weekly,No,PayPal,Neutral,39.0,3.76,58.0,27.0,0.678,0.667,0.0,26
4,Male,Bi-Weekly,No,Cash,Neutral,46.0,3.72,60.0,27.0,0.687,0.671,0.0,26
...,...,...,...,...,...,...,...,...,...,...,...,...,...
64,Female,Annually,No,Debit Card,Neutral,53.0,3.20,96.0,47.0,0.613,0.957,0.0,1
65,Female,Annually,No,Venmo,Highly Satisfied,52.0,4.40,63.0,50.0,0.534,1.000,0.0,1
66,Female,Annually,No,Cash,Highly Satisfied,57.0,4.40,64.0,44.0,0.489,0.914,0.0,1
67,Male,Annually,No,Debit Card,Highly Satisfied,54.0,5.00,95.0,40.0,0.553,0.857,0.0,1


### **Q5b - ICP Shipping Type, Size & Item Preferences**

In [141]:
# Q5b-i: Shipping type preference for ICP customers
result_q5b_shipping = con.sql("""
SELECT
    "Shipping Type",
    COUNT(*)                                   AS icp_customers,
    ROUND(COUNT(*)*100.0/SUM(COUNT(*)) OVER(), 1) AS pct_of_icp,
    ROUND(AVG(clv_score), 3)                   AS avg_clv,
    ROUND(AVG(loyalty_score), 3)               AS avg_loyalty_score,
    ROUND(AVG("Purchase Amount (USD)"), 2)     AS avg_spend,
    ROUND(AVG(dependency_score), 3)            AS avg_promo_dep
FROM df
WHERE
    full_price_buyer = 1
    AND clv_tier = 'High'
    AND consistent = 1
GROUP BY "Shipping Type"
ORDER BY icp_customers DESC
""").df()
display(result_q5b_shipping)

,Shipping Type,icp_customers,pct_of_icp,avg_clv,avg_loyalty_score,avg_spend,avg_promo_dep
0,2-Day Shipping,123,17.5,0.611,0.771,73.54,0.0
1,Free Shipping,119,17.0,0.632,0.773,68.56,0.0
2,Express,119,17.0,0.622,0.776,69.27,0.0
3,Store Pickup,118,16.8,0.634,0.765,73.34,0.0
4,Next Day Air,112,16.0,0.628,0.781,68.65,0.0
5,Standard,110,15.7,0.627,0.771,68.46,0.0


In [142]:
# Q5b-ii: Size preference for ICP customers
result_q5b_size = con.sql("""
SELECT
    Size,
    COUNT(*)                                      AS icp_customers,
    ROUND(COUNT(*)*100.0/SUM(COUNT(*)) OVER(), 1) AS pct_of_icp,
    ROUND(AVG(clv_score), 3)                      AS avg_clv,
    ROUND(AVG(loyalty_score), 3)                  AS avg_loyalty_score,
    ROUND(AVG("Purchase Amount (USD)"), 2)         AS avg_spend
FROM df
WHERE
    full_price_buyer = 1
    AND clv_tier = 'High'
    AND consistent = 1
GROUP BY Size
ORDER BY icp_customers DESC
""").df()
display(result_q5b_size)

,Size,icp_customers,pct_of_icp,avg_clv,avg_loyalty_score,avg_spend
0,M,311,44.4,0.629,0.762,72.30
1,L,183,26.1,0.621,0.796,67.92
2,S,129,18.4,0.617,0.774,67.49
3,XL,78,11.1,0.632,0.759,73.09


In [143]:
# Q5b-iii: Top 10 items purchased by ICP customers
result_q5b_items = con.sql("""
SELECT
    "Item Purchased",
    Category,
    COUNT(*)                                      AS icp_customers,
    ROUND(COUNT(*)*100.0/SUM(COUNT(*)) OVER(), 1) AS pct_of_icp,
    ROUND(AVG("Purchase Amount (USD)"), 2)         AS avg_spend,
    ROUND(AVG(loyalty_score), 3)                  AS avg_loyalty_score,
    ROUND(AVG(clv_score), 3)                      AS avg_clv
FROM df
WHERE
    full_price_buyer = 1
    AND clv_tier = 'High'
    AND consistent = 1
GROUP BY "Item Purchased", Category
ORDER BY icp_customers DESC
LIMIT 10
""").df()
display(result_q5b_items)

,Item Purchased,Category,icp_customers,pct_of_icp,avg_spend,avg_loyalty_score,avg_clv
0,Blouse,Clothing,40,5.7,71.60,0.759,0.635
1,Jewelry,Accessories,37,5.3,68.54,0.846,0.668
2,Scarf,Accessories,37,5.3,74.30,0.733,0.606
3,Sunglasses,Accessories,33,4.7,69.94,0.750,0.602
4,Jacket,Outerwear,33,4.7,69.39,0.761,0.600
5,Shorts,Clothing,32,4.6,64.75,0.772,0.638
6,Gloves,Accessories,31,4.4,68.32,0.790,0.632
7,Dress,Clothing,31,4.4,71.71,0.771,0.623
8,Sweater,Clothing,31,4.4,64.10,0.739,0.612
9,Pants,Clothing,30,4.3,69.93,0.758,0.625


# **Engagement Priority Matrix**
*Strategic matrix combining CLV tier and purchase consistency to prioritise customer engagement and value-preservation efforts. Revenue represented provides context on the economic significance of each segment.*

In [144]:
# High Value-Stable = core base, highest single-period spend ($86,631), strong loyalty (0.652), low promo dependency (0.163).
# Mid Value-Behaviorally Inconsistent = largest engagement opportunity (610 customers), elevated promo dependency (0.298).
# High Value-Behaviorally Inconsistent = small (23 customers) but high spend/customer
# ($95.96) and highest promo dependency (0.345) -> top priority for retention/discount optimization.

In [148]:
result_epm = con.sql("""
    SELECT
        CASE
            WHEN clv_tier='High' AND consistent=0 THEN 'High Value - Behaviorally Inconsistent'
            WHEN clv_tier='High' AND consistent=1 THEN 'High Value - Stable'
            WHEN clv_tier='Mid'  AND consistent=0 THEN 'Mid Value - Behaviorally Inconsistent'
            WHEN clv_tier='Mid'  AND consistent=1 THEN 'Mid Value - Stable'
            ELSE 'Low Value'
        END                                         AS engagement_priority,
        COUNT(*)                                    AS customers,
        ROUND(AVG(clv_score), 3)                    AS avg_clv,
        ROUND(AVG(loyalty_score), 3)                AS avg_loyalty_score,
        ROUND(AVG(dependency_score), 3)             AS avg_promo_dep,
        ROUND(AVG("Purchase Amount (USD)"), 2)      AS avg_spend,
        ROUND(SUM("Purchase Amount (USD)"), 0)      AS single_period_spend  -- sum of one purchase per customer row
    FROM df
    GROUP BY engagement_priority
    ORDER BY single_period_spend DESC
""").df()
display(result_epm)

,engagement_priority,customers,avg_clv,avg_loyalty_score,avg_promo_dep,avg_spend,single_period_spend
0,High Value - Stable,1237,0.626,0.652,0.163,70.03,86631.0
1,Low Value,1260,0.219,0.338,0.361,49.73,62661.0
2,Mid Value - Behaviorally Inconsistent,610,0.385,0.469,0.298,75.80,46241.0
3,Mid Value - Stable,650,0.420,0.626,0.243,42.80,27817.0
4,High Value - Behaviorally Inconsistent,23,0.503,0.537,0.345,95.96,2207.0


# **Data Quality Audit: Imputed Review Ratings** 
*Assessing the impact of 36 imputed Review Ratings on satisfaction-based analysis using the `rating_missing` indicator.*

In [149]:
# Review-rating data quality is very strong, with only 36 missing ratings (0.95%) across the entire customer base.
# Missing values are limited to promotion-dependent customers and have no meaningful impact on average ratings, satisfaction segments, 
# loyalty scores, or CLV metrics. The analysis remains reliable, and no business conclusions change after accounting for the imputed ratings.

In [150]:
#Distribution of rating_missing by customer segment
result_dq_i = con.sql("""
SELECT
    customer_segment,
    COUNT(*)                                           AS total_customers,
    SUM(rating_missing)                                AS rows_with_imputed_rating,
    ROUND(SUM(rating_missing)*100.0/COUNT(*), 2)       AS imputed_pct,
    ROUND(AVG(CASE WHEN rating_missing=0 THEN "Review Rating" END), 3) AS avg_rating_observed,
    ROUND(AVG(CASE WHEN rating_missing=1 THEN "Review Rating" END), 3) AS avg_rating_imputed,
    ROUND(AVG("Review Rating"), 3)                     AS avg_rating_combined
FROM df
GROUP BY customer_segment
ORDER BY imputed_pct DESC
""").df()
display(result_dq_i)


,customer_segment,total_customers,rows_with_imputed_rating,imputed_pct,avg_rating_observed,avg_rating_imputed,avg_rating_combined
0,Loyal: Promo Dependent,831,19.0,2.29,3.729,3.732,3.729
1,Casual: Bargain Hunter,797,17.0,2.13,3.754,3.759,3.754
2,Casual: Organic,1079,0.0,0.00,3.756,NaN,3.756
3,Loyal: Brand Purist,1073,0.0,0.00,3.756,NaN,3.756


In [151]:
#Impact on satisfaction_tier composition with vs without imputed rows
result_dq_ii = con.sql("""
SELECT
    satisfaction_tier,
    COUNT(*)                                           AS all_customers,
    COUNT(CASE WHEN rating_missing=0 THEN 1 END)       AS observed_rating_customers,
    COUNT(CASE WHEN rating_missing=1 THEN 1 END)       AS imputed_rating_customers,
    ROUND(COUNT(CASE WHEN rating_missing=1 THEN 1 END)*100.0/COUNT(*), 2) AS imputed_pct,
    ROUND(AVG(clv_score), 3)                           AS avg_clv,
    ROUND(AVG(loyalty_score), 3)                       AS avg_loyalty_score
FROM df
GROUP BY satisfaction_tier
ORDER BY all_customers DESC
""").df()
display(result_dq_ii)


,satisfaction_tier,all_customers,observed_rating_customers,imputed_rating_customers,imputed_pct,avg_clv,avg_loyalty_score
0,Neutral,1552,1516,36,2.32,0.413,0.512
1,Highly Satisfied,1411,1411,0,0.00,0.417,0.515
2,Dissatisfied,817,817,0,0.00,0.417,0.508


In [152]:
#rating_missing by loyalty_tier and clv_tier
result_dq_iii = con.sql("""
SELECT
    loyalty_tier,
    clv_tier,
    COUNT(*)                                           AS customers,
    SUM(rating_missing)                                AS imputed_rows,
    ROUND(SUM(rating_missing)*100.0/COUNT(*), 2)       AS imputed_pct
FROM df
GROUP BY loyalty_tier, clv_tier
ORDER BY imputed_pct DESC
""").df()
display(result_dq_iii)


,loyalty_tier,clv_tier,customers,imputed_rows,imputed_pct
0,Low,High,173,6.0,3.47
1,Mid,Mid,455,8.0,1.76
2,Mid,High,411,7.0,1.70
3,Low,Mid,328,4.0,1.22
4,Low,Low,771,9.0,1.17
5,Mid,Low,409,1.0,0.24
6,High,High,676,1.0,0.15
7,High,Low,80,0.0,0.00
8,High,Mid,477,0.0,0.00


# **Summary**

## 1. Who are the genuinely loyal customers vs. those who only buy when there is a discount?

Two distinct groups emerge. *Loyal: Brand Purist* customers are the clearest case of genuine loyalty - they post the highest loyalty score (0.767) and zero promotional dependency, meaning their repeat purchasing happens with no discount incentive at all. *Loyal: Promo Dependent* customers, by contrast, generate comparable CLV (0.552 vs. 0.554) but carry meaningfully higher dependency (0.438) and lower loyalty (0.479) - the same revenue, sustained by a different (and riskier) mechanism. The loyalty-tier breakdown confirms this gradient holds across the whole base: High-loyalty customers average just 0.049 dependency, versus 0.208 (Mid) and 0.533 (Low).

`loyalty_score` (Definition B - margin loyalty: 0.7 × engagement + 0.3 × full-price behavior) was adopted over Definition A (behavioral: engagement + frequency + subscription) because Definition A's top quartile was 81% discount users and 77% subscribers - it was effectively rewarding discount-access behavior, not commitment, and was largely redundant with `commitment_factor` (r ~ 0.9). Definition B is not fully independent of `clv_score` - both draw on `engagement_normalized` - but unlike Definition A it isn't contaminated by subscription status, making it the more defensible lens for *genuine* loyalty versus *revenue potential* (`clv_score`).

---

## 2. What behavioral patterns today predict high customer value over time?

**Purchase frequency** is the dominant driver of CLV: Bi-Weekly buyers post both the highest average CLV (0.671) and the lowest promotional dependency (0.103), and value declines steadily as purchase frequency drops - spend and rating stay roughly flat across frequency groups, so frequency (via `engagement_normalized`/`frequency_normalized`) is doing the work, not transaction size.

**Subscription status shows only a weak, indirect link to value.** Subscribers post a marginally higher average CLV (0.421 vs. 0.413), but `clv_score` excludes `subscription_binary` by design - the correlation between the two is negligible (r ~ 0.017). What subscription status *does* predict strongly is discount usage (r ~ -0.70 with full-price buying): subscription functions as a discount-access mechanism rather than a value or loyalty signal in this dataset.

**Loyalty and dependency move in lockstep**, as shown in Q1b - high-loyalty customers (0.049 dependency) are essentially promotion-independent. Taken together, the strongest value predictors are *purchase frequency* and *promotion-independent loyalty*; subscription is a weak, confounded signal that should not be used as a standalone targeting variable.

---

## 3. Which geographies and demographics are commercially underlevered?

* **Geographically**, Arizona (54.17), Alaska (51.74), and Tennessee (49.43) post the strongest Organic Demand Proxy scores - above-average spend combined with below-average promotional dependency - making them priority markets for acquisition and retention investment.

* **Demographically**, Q3b surfaces Male, 46-55 as the highest-CLV segment (0.433), but this finding carries an important caveat: in this dataset, **Gender is fully confounded with discount usage** - every female customer is a full-price buyer (`full_price_buyer` = 1, `dependency_score` = 0), and all discount and subscription activity is concentrated among male customers. Apparent gender differences in CLV/dependency therefore reflect this structural artifact rather than a real demographic preference, and gender should not be used as a standalone targeting variable until validated against a less confounded dataset.

* **Item-level analysis (Q3c)** shows product choice is a stronger value signal than seasonality: items with high `organic_purchase_strength` (high CLV × low dependency) are natural candidates for retention and full-price growth campaigns, while items skewed toward off-season or discount-heavy purchases (e.g., winter items bought in summer) show elevated promo dependency and are better suited to clearance-style, time-bound offers.

---

## 4. How should the brand restructure its promotional strategy to protect margins without losing volume?

* **Promo dependency is concentrated in two segments** - *Loyal: Promo Dependent* and *Casual: Bargain Hunter*. Q4a's head-to-head comparison shows *Loyal: Promo Dependent* customers behave almost identically to *Brand Purists* on spend, cadence, and satisfaction, differing mainly in discount usage - suggesting the discount has become a habit rather than a purchasing necessity for this group, and is a credible sunset target.

* **Sunset candidates**: 662 *Loyal: Promo Dependent* customers with High/Mid CLV and 25+ previous purchases. The discount gap between Brand Purists (\\$60.73 avg spend) and Promo Dependents (\\$59.83) is small (~\\$0.89/txn, ~1.5%) - meaningful in aggregate across this group's purchase volume but modest per customer, so the financial case should be framed as a **lower-bound, conservative estimate** rather than the full margin opportunity (Q4e: ~\\$16K-21K/year across the three churn scenarios).

* **The store-credit framework (Q4b-Q4d)** converts this gap into an actionable system: credit ceilings calibrated to each segment's discount value, expiry windows set at 2× each segment's natural purchase interval (so credits are redeemable within a normal shopping cycle), and earn rates calibrated so one cycle's spend crosses the redemption threshold.

* **Recommended rollout**: phase out blanket discounts for the 662 sunset candidates over a defined window, replacing them with the store-credit mechanism above; track `dependency_score` and repeat-purchase rate for this cohort monthly to confirm the transition doesn't trigger the churn levels assumed in the conservative scenario. Casual: Bargain Hunter customers are not included in this first wave - their behavioral profile (Q4a/Q4c) differs enough from Brand Purists that the "habit not necessity" argument doesn't yet hold for them, and they warrant a separate, lower-risk pilot.

---

## 5. What does the brand's ideal customer profile look like, and how can it acquire more of them?

### Ideal Customer Profile (ICP)

* High-CLV (`clv_tier` = High), full-price (`full_price_buyer` = 1), behaviorally consistent (`consistent` = 1) customer.
* Typically shops bi-weekly, with 30+ previous purchases (segment average ~ 34, median = 38).
* Spends roughly $70-75 per order - above the overall customer average.
* Median age is 45, with no strong age group skew within the segment.
* Predominantly non-subscribers and disproportionately female - **both are dataset structure effects** (every subscriber has `discount_flag` = 1, and every female customer has `full_price_buyer` = 1), not evidence that non-subscription or female gender independently drive loyalty.
* Prefers M (45%) and L (26%) sizes - over 70% of the ICP combined.
* Top items: Jewelry, Blouses, Scarves, Gloves, and Dresses - accessories and everyday staples rather than seasonal pieces.
* Mostly Neutral or Highly Satisfied - a stable, self-sustaining segment with no discount dependency.

**Acquisition implication**: target full-price-leaning, high-frequency shoppers with an affinity for accessories/staples rather than seasonal categories, and avoid using subscription status or gender as targeting filters given the confounds above.

---

## Engagement Priority Matrix

Cross-tabulating CLV tier with purchase consistency produces five priority segments, each requiring a different lever:

* **High Value - Stable** (core base): highest single-period spend ($86,631), strong loyalty (0.652), low promo dependency (0.163) - protect via loyalty benefits and exclusives, not discounts.

* **Mid Value - Behaviorally Inconsistent** (610 customers, largest segment): elevated promo dependency (0.298) - the biggest conversion opportunity; retention programs and personalization aimed at shifting these customers toward "Stable" status could meaningfully grow the loyal base.

* **High Value - Behaviorally Inconsistent** (23 customers, small but high-impact): highest spend per customer ($95.96) and highest promo dependency (0.345) - prioritize for personalized retention offers and gradual discount reduction, given their outsized per-customer value.

* **Low Value segments**: limited revenue contribution; lower marketing priority unless acquisition cost is minimal.

---

## Data Quality Audit (`rating_missing`)

Only 36 ratings (0.95%) were missing and imputed via frequency-group median. Imputed records fall entirely within the Neutral satisfaction tier and are concentrated among promotion-dependent customers; satisfaction tier composition, loyalty scores, CLV estimates, and segment profiles are materially unchanged with or without these rows. The imputation does not affect any conclusion above.

---

## Final Recommendations

1. **Protect High Value - Stable customers** via loyalty benefits, early access, and personalized engagement - not discounts.
2. **Pilot the promotional sunset** on the 662 *Loyal: Promo Dependent* / High-Mid CLV / 25+ purchase candidates, replacing discounts with the store-credit system (Q4b-Q4d), and track `dependency_score` monthly against the conservative-scenario assumption.
3. **Prioritize conversion of Mid Value - Behaviorally Inconsistent customers** (610, the largest segment) toward "Stable" status through retention and personalization.
4. **Prioritize acquisition in Arizona, Alaska, and Tennessee**, the top Organic Demand Proxy states.
5. **Use `loyalty_score` (Definition B) as the primary lens for genuine commitment**, and `clv_score` for revenue potential - they are related but not interchangeable, and neither should be read in isolation from `dependency_score`.
6. **Align acquisition and merchandising with the ICP** (accessories/staples, M/L sizes, high-frequency shoppers) while avoiding gender and subscription status as targeting filters, given the confounds documented in Q3b and the ICP note.

---

## Strategic Bottom Line

The brand's strongest customers are not its highest spenders but its most *consistent, promotion-independent* ones - Brand Purists and Promo Dependents generate near-identical revenue, but only one carries promotional risk. The path forward is a loyalty-led strategy: convert the large Mid-Value/Inconsistent base toward Stable status, sunset discounts for customers who no longer need them (starting with the 662 identified candidates), and replace blanket promotions with a calibrated store-credit system that preserves the redemption "feel" of a discount without the margin leakage.

